# Train Unified SFT Model (Hugging Face Cloud integration)

This notebook clones the latest code repository from GitHub and executes SFT training. The dataset is loaded directly from Hugging Face Hub, and checkpoints are continuously pushed/updated to the HF Hub repository during training.

In [1]:
import os
import sys
import torch
from pathlib import Path
import argparse
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    set_seed
)
from peft import LoraConfig, TaskType
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from huggingface_hub import login

# Safe repo cloning and directory changing
repo_url = "https://github.com/NguyenAn05/exact-2026.git"
repo_dir = "exact-2026"

if Path(os.getcwd()).name == repo_dir:
    print(f"Already inside the repository directory: {os.getcwd()}")
else:
    if not os.path.exists(repo_dir):
        print(f"Cloning repository from {repo_url}...")
        os.system(f"git clone {repo_url}")
    os.chdir(repo_dir)
    print(f"Changed working directory to: {os.getcwd()}")

print(f"Current working directory: {os.getcwd()}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Changed working directory to: /workspace/exact-2026
Current working directory: /workspace/exact-2026
CUDA Available: True
GPU: AMD Instinct MI300X VF


In [2]:
def parse_args():
    parser = argparse.ArgumentParser(description="Unified SFT Training for logic reasoning model (HF Cloud).")
    parser.add_argument("--model_id", type=str, default="Qwen/Qwen2.5-7B-Instruct")
    parser.add_argument("--dataset_id", type=str, default="NguyenAn05/exact_2026_model1_fol_translator")
    parser.add_argument("--output_dir", type=str, default="models/qwen2.5-type1-sft-lora")
    parser.add_argument("--hub_model_id", type=str, default="NguyenAn05/qwen2.5-type1-sft-lora")
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--grad_accum", type=int, default=2)
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--lr", type=float, default=2e-5)
    parser.add_argument("--max_seq_len", type=int, default=4096)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--max_steps", type=int, default=-1)
    parser.add_argument("--save_steps", type=int, default=20)
    
    # Check if running in a Jupyter notebook/IPython session
    is_ipython = False
    try:
        from IPython import get_ipython
        if get_ipython() is not None:
            is_ipython = True
    except ImportError:
        pass
        
    if is_ipython or any('ipykernel' in arg for arg in sys.argv):
        return parser.parse_args(args=[])
    else:
        return parser.parse_args()

In [3]:
def main():
    args = parse_args()
    set_seed(args.seed)

    # Login to Hugging Face Hub (required for writing/pushing model)
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    login(token=HF_TOKEN)

    print(f"CUDA Available: {torch.cuda.is_available()}")
    print(f"Device Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

    # Load Tokenizer
    print(f"Loading tokenizer for {args.model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(args.model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # Load dataset from Hugging Face Hub
    print(f"Loading dataset from HF Hub: {args.dataset_id}...")
    dataset = load_dataset(args.dataset_id)

    def format_sample(example):
        return {
            "text": tokenizer.apply_chat_template(
                example["messages"],
                tokenize=False,
                add_generation_prompt=False
            )
        }

    dataset = dataset.map(format_sample, remove_columns=["messages"])

    # Load Model in bfloat16
    print(f"Loading base model {args.model_id} in bfloat16...")
    model = AutoModelForCausalLM.from_pretrained(
        args.model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="flash_attention_2"
    )

    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False

    # Set up LoRA Configuration
    peft_config = LoraConfig(
        r=64,
        lora_alpha=128,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )

    # Setup Data Collator for Completion Only Training
    response_template_ids = tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
    collator = DataCollatorForCompletionOnlyLM(
        response_template=response_template_ids,
        tokenizer=tokenizer
    )

    # Training Arguments with active checkpoint uploading to HF Hub
    use_steps = args.max_steps < 0
    training_args = TrainingArguments(
        output_dir=args.output_dir,
        per_device_train_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        num_train_epochs=args.epochs,
        max_steps=args.max_steps,
        bf16=True,
        fp16=False,
        logging_steps=10,
        eval_strategy="steps" if use_steps else "no",
        eval_steps=args.save_steps,
        save_strategy="steps" if use_steps else "no",
        save_steps=args.save_steps,
        push_to_hub=True,                   # Enable continuous push to HF hub
        hub_model_id=args.hub_model_id,
        hub_strategy="checkpoint",           # Push checkpoints continuously
        load_best_model_at_end=True if use_steps else False,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=3,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        optim="adamw_torch_fused",
        report_to="none",
        gradient_checkpointing=True,
        ddp_find_unused_parameters=False
    )

    # Initialize SFTTrainer
    print("Initializing SFTTrainer...")
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"] if "validation" in dataset else (dataset["val"] if "val" in dataset else dataset["test"]),
        peft_config=peft_config,
        tokenizer=tokenizer,
        data_collator=collator,
        dataset_text_field="text",
        args=training_args,
        max_seq_length=args.max_seq_len
    )

    # Run SFT Training
    print("\n--- Starting SFT Training ---")
    trainer.train()

    # Save the final adapter locally
    print(f"Saving final LoRA adapter locally to {args.output_dir}...")
    trainer.model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

    # Final push to HF Hub
    print("\nPushing best LoRA adapter to Hugging Face Hub...")
    trainer.push_to_hub()
        
    print("Training and HF uploading complete!")

if __name__ == "__main__":
    main()

CUDA Available: True
Device Name: AMD Instinct MI300X VF
Loading tokenizer for Qwen/Qwen2.5-7B-Instruct...
Loading dataset from HF Hub: NguyenAn05/exact_2026_model1_fol_translator...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model Qwen/Qwen2.5-7B-Instruct in bfloat16...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Initializing SFTTrainer...


...

Step,Training Loss,Validation Loss
20,0.410500,0.345841
40,0.243400,0.214706
60,0.179200,0.172571
80,0.132400,0.145076
100,0.119500,0.124095
120,0.101700,0.106523
140,0.083000,0.090721
160,0.083600,0.077732
180,0.057500,0.069065
200,0.050900,0.063169


Saving final LoRA adapter locally to models/qwen2.5-type1-sft-lora...

Pushing best LoRA adapter to Hugging Face Hub...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Training and HF uploading complete!
